# Etapa 2d — Mapas de color con el fondo corregido

**Por qué.** En la etapa 3, la corrección del fondo llevó al verde de 0,687 a 0,941 y lo dejó muy por encima de la tercera componente (0,868 con su mejor preprocesamiento). La elección de la etapa 2 comparó mapas crudos, donde el verde pierde por la iluminación despareja. *Desvío declarado:* el mapa y el preprocesamiento se deciden juntos; se rehace la elección.

**Qué se compara.** Los diez mapas, cada uno con su versión de mejoramiento que mejor le funcione entre las dos que ganaron en la etapa 3 (**corrección del fondo** y **ecualización adaptativa**), precedidas por el suavizado de bloques y Wiener contra el ruido, que se adoptaron en la etapa 3. Después, combinaciones lineales (discriminante de Fisher) del mejor mapa con cada uno de los otros y de los diez juntos, ajustadas con las personas de entrenamiento de cada vuelta y medidas en las de afuera.

**Cómo se mide.** Con las mejoras de velocidad (`src/evaluation.py`): imágenes en paralelo y área bajo la curva en la muestra fija de evaluación (todos los vasos y el 20 % del fondo, a resolución original). Se compara contra los valores de la etapa 3 para ver el error de la muestra.

**Criterios** (los mismos de la etapa 2): mejor mapa por área media; no inferioridad con margen −0,02; estabilidad con validación cruzada por personas; pasan a la combinación los mapas que le suman al menos 0,01 al mejor.


In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.color import MAP_NAMES
from src.data import dev_ids, load_split
from src.split import person_folds
from src.stats import fisher_direction, percentile_interval, person_bootstrap_means

ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
PREPS = ["fondo", "ecualización"]
N_TRAIN = 20_000   # píxeles de vaso y de fondo por imagen para ajustar combinaciones

In [2]:
def corrected_maps(image_id, preps=None):
    """Los diez mapas a la mitad de resolución, con bloques + Wiener contra el ruido + mejoramiento.

    Devuelve {(mapa, mejoramiento): mapa estandarizado dentro del campo detectado}.
    `preps` = {mapa: mejoramiento} limita a uno por mapa.
    """
    import numpy as np
    from PIL import Image
    from src import config as C
    from src.color import MAP_NAMES, color_maps, standardize
    from src.data import load_image
    from src.restoration import background_correct, clahe, deblock, wiener_denoise
    from src.fov import detect_fov_robust, fov_diameter

    rgb = load_image(image_id)
    small = np.asarray(Image.fromarray(deblock(rgb)).reduce(C.STAGE3_FACTOR))
    region = detect_fov_robust(small)
    size = int(round(C.BACKGROUND_WINDOW_REL * fov_diameter(region))) | 1
    out = {}
    for name, m in color_maps(small, region).items():
        m = wiener_denoise(m, region, C.WIENER_SIZE)
        for prep in (C_PREPS if preps is None else [preps[name]]):
            x = background_correct(m, region, size) if prep == "fondo" else clahe(m, region, C.CLAHE_CLIP, C.CLAHE_KERNEL_FRAC)
            out[(name, prep)] = standardize(x, region)
    return out

C_PREPS = PREPS

def pass1(image_id):
    import numpy as np
    from src.evaluation import load_sample, sample_auc, sample_values
    t0 = time.time()
    s = load_sample(image_id)
    maps = corrected_maps(image_id)
    rng = np.random.default_rng(C.SEED + len(image_id) + sum(map(ord, image_id)))
    pos, neg = np.flatnonzero(s.truth), np.flatnonzero(~s.truth)
    pick = np.r_[rng.choice(pos, N_TRAIN, replace=False), rng.choice(neg, N_TRAIN, replace=False)]
    rows, feats = [], {}
    for (name, prep), m in maps.items():
        v = sample_values(m, s)
        a = sample_auc(m, s)
        rows.append({"image": image_id, "map": name, "prep": prep, "todos": a["todos"], "finos": a["finos"]})
        feats[(name, prep)] = v[pick].astype(np.float32)
    return rows, feats, s.truth[pick], time.time() - t0

t = time.time()
out1 = __import__("src.evaluation", fromlist=["map_images"]).map_images(pass1, ids, workers=4)
print(f"Pasada 1: {time.time() - t:.0f} s para 30 imágenes; {np.mean([o[3] for o in out1]):.1f} s por imagen dentro de cada proceso")
R = pd.DataFrame([r for o in out1 for r in o[0]])
R.to_csv(ROOT / "results" / "02d_mapas_corregidos.csv", index=False)

Pasada 1: 508 s para 30 imágenes; 64.6 s por imagen dentro de cada proceso


## Error de la muestra frente a todos los píxeles

La tercera componente y el verde con corrección del fondo ya se midieron en la etapa 3 con todos los píxeles (sin bloques ni Wiener). Para comparar, se toma la misma cadena de la etapa 3 (`bloques → Wiener ruido → fondo`) de la pasada 2 de la etapa 3.

In [3]:
p3 = pd.read_csv(ROOT / "results" / "03_pasada2.csv")
cmp_rows = []
for name in ("PC3", "Verde"):
    e = "ecualización" if name == "PC3" else "fondo"
    ref = p3[(p3["map"] == name) & (p3["cand"] == f"bloques → Wiener ruido → {e}")].set_index("image")["todos"]
    mine = R[(R["map"] == name) & (R["prep"] == e)].set_index("image")["todos"]
    d = (mine - ref).dropna()        # las dos son áreas con vasos como valores altos, sin orientar
    cmp_rows.append({"mapa": name, "cadena": f"bloques → Wiener ruido → {e}", "imágenes": len(d),
                     "diferencia media (muestra − todos)": d.mean(), "máxima absoluta": d.abs().max()})
pd.DataFrame(cmp_rows).round(5)

,mapa,cadena,imágenes,diferencia media (muestra − todos),máxima absoluta
0,PC3,bloques → Wiener ruido → ecualización,30,0.00003,0.00040
1,Verde,bloques → Wiener ruido → fondo,30,-0.00002,0.00016


## Ranking de los diez mapas corregidos

In [4]:
def oriented_table(R, prep_choice=None, measure="todos"):
    w = R.pivot_table(index="image", columns=["map", "prep"], values=measure)
    flip = w.mean() < 0.5
    w.loc[:, flip] = 1 - w.loc[:, flip]
    return w

W = oriented_table(R)
best_prep = {m: W[m].mean().idxmax() for m in MAP_NAMES}
Wb = pd.DataFrame({m: W[(m, best_prep[m])] for m in MAP_NAMES})
Fb = oriented_table(R, measure="finos")
Fb = pd.DataFrame({m: Fb[(m, best_prep[m])] for m in MAP_NAMES})
best = Wb.mean().idxmax()
diffs = Wb.sub(Wb[best], axis=0)
lo, hi = percentile_interval(person_bootstrap_means(diffs, dev["person"]))
rank = pd.DataFrame({"mejoramiento": pd.Series(best_prep), "área": Wb.mean(), "área finos": Fb.mean(),
                     "peor imagen": Wb.min(), "vs mejor": diffs.mean(), "límite inferior": lo,
                     "no inferior": lo >= -C.NONINF_MARGIN_AUC}).sort_values("área", ascending=False)
rank.round(4)

,mejoramiento,área,área finos,peor imagen,vs mejor,límite inferior,no inferior
Verde,fondo,0.9417,0.8457,0.8813,0.0000,0.0000,True
Escala de grises,fondo,0.9315,0.8182,0.8721,-0.0102,-0.0131,True
PC2,fondo,0.9152,0.7968,0.8475,-0.0265,-0.0349,False
Lab L*,fondo,0.8954,0.7490,0.8095,-0.0462,-0.0585,False
PC1,fondo,0.8746,0.7211,0.7628,-0.0670,-0.0846,False
PC3,fondo,0.8736,0.7216,0.7845,-0.0680,-0.0799,False
Lab a*,fondo,0.8354,0.7182,0.6904,-0.1063,-0.1285,False
Lab b*,fondo,0.8133,0.6623,0.6219,-0.1284,-0.1729,False
Rojo,fondo,0.7858,0.6343,0.6621,-0.1559,-0.1867,False
Azul,fondo,0.7114,0.6326,0.4333,-0.2303,-0.2821,False


In [5]:
picks = []
for k in range(5):
    tr = folds.index[folds != k]
    picks.append(Wb.loc[tr].mean().idxmax())
print("Mejor mapa por vuelta:", picks)

Mejor mapa por vuelta: ['Verde', 'Verde', 'Verde', 'Verde', 'Verde']


## Combinaciones con el mejor mapa (pasada 2)

In [6]:
prep_of = best_prep
feat_cols = MAP_NAMES
X_by_img = {o[0][0]["image"]: np.stack([o[1][(m, prep_of[m])] for m in feat_cols], axis=1) for o in out1}
y_by_img = {o[0][0]["image"]: o[2] for o in out1}
ib = feat_cols.index(best)
best_sign = 1 if R[(R["map"] == best) & (R["prep"] == prep_of[best])]["todos"].mean() >= 0.5 else -1
COMBOS = {f"{best} + {m}": [ib, feat_cols.index(m)] for m in feat_cols if m != best}
COMBOS["los diez"] = list(range(len(feat_cols)))
weights = {}
for k in range(5):
    tr = folds.index[folds != k]
    X = np.concatenate([X_by_img[i] for i in tr]); y = np.concatenate([y_by_img[i] for i in tr])
    for combo, cols in COMBOS.items():
        weights[(k, combo)] = fisher_direction(X[:, cols], y)

def pass2(image_id):
    from src.evaluation import load_sample, sample_values
    from src.metrics import auc_by_subset
    s = load_sample(image_id)
    maps = corrected_maps(image_id, prep_of)
    V = np.stack([sample_values(maps[(m, prep_of[m])], s) for m in feat_cols], axis=1)
    k = int(folds[image_id])
    row = {"image": image_id, f"{best} solo": auc_by_subset(best_sign * V[:, ib], s.truth, s.subsets())["todos"]}
    for combo, cols in COMBOS.items():
        row[combo] = auc_by_subset(V[:, cols] @ weights[(k, combo)], s.truth, s.subsets())["todos"]
    return row

t = time.time()
Cmb = pd.DataFrame(__import__("src.evaluation", fromlist=["map_images"]).map_images(pass2, ids, workers=4)).set_index("image")
print(f"Pasada 2: {time.time() - t:.0f} s")
Cmb.to_csv(ROOT / "results" / "02d_combinaciones.csv")
gain = Cmb.drop(columns=f"{best} solo").sub(Cmb[f"{best} solo"], axis=0)
lo, hi = percentile_interval(person_bootstrap_means(gain, dev["person"]))
comp = pd.DataFrame({"área combinada": Cmb.drop(columns=f"{best} solo").mean(), "ganancia": gain.mean(),
                     "límite inferior": lo, "límite superior": hi}).sort_values("ganancia", ascending=False)
print(f"{best} solo, personas de afuera: {Cmb[f'{best} solo'].mean():.4f}")
comp.round(4)

Pasada 2: 278 s
Verde solo, personas de afuera: 0.9417


,área combinada,ganancia,límite inferior,límite superior
Verde + Lab a*,0.9431,0.0014,0.0007,0.0023
Verde + PC2,0.9425,0.0009,0.0005,0.0013
Verde + Rojo,0.9422,0.0005,0.0000,0.0011
Verde + PC1,0.9421,0.0005,-0.0001,0.0011
Verde + Azul,0.9420,0.0004,-0.0005,0.0016
Verde + Lab L*,0.9420,0.0004,-0.0002,0.0009
Verde + Escala de grises,0.9416,-0.0000,-0.0011,0.0011
Verde + Lab b*,0.9415,-0.0002,-0.0003,-0.0001
Verde + PC3,0.9414,-0.0003,-0.0015,0.0013
los diez,0.9410,-0.0007,-0.0026,0.0016


## Decisión (escrita después de correr)

**La muestra de evaluación es fiel:** frente a usar todos los píxeles, el área difiere en 0,00003 de media y 0,0004 como máximo. La corrida tardó unos 13 minutos en total para 20 mapas por imagen (la etapa 3 tardó casi una hora para 2).

**Con el fondo corregido, el mejor mapa es el verde** (0,942; vasos finos 0,846), elegido en las 5 vueltas de la validación cruzada. La escala de grises queda cerca y es no inferior (0,932); todos los demás son claramente peores, incluida la tercera componente (0,874). Para todos los mapas el mejor mejoramiento es la corrección del fondo.

**Combinar no suma:** ningún mapa le agrega al verde más de 0,0014 de área, y los diez juntos dan 0,941, lo mismo que el verde solo. Ninguno llega al corte de 0,01.

**Se cambia la elección de la etapa 2** (desvío declarado en el esquema):
- **Mapa principal: verde**, con bloques → Wiener contra el ruido → corrección del fondo (etapa 3).
- **Mapas complementarios: ninguno** a nivel de intensidad.
- **Referencia clásica: el mismo verde**, así que la diferencia entre el algoritmo final y la referencia (pregunta principal del plan) tiene que venir de combinar varios **mapas de características**: filtros de realce a distintas escalas y orientaciones (etapa 4) y textura, no de cambiar de canal de color.

**Qué se informa sobre el color.** La ventaja de la tercera componente sobre el verde en mapas crudos (0,81 contra 0,69) se debía a la iluminación despareja: una vez corregida, el color no agrega información de intensidad sobre los vasos. Es un resultado negativo que responde la sugerencia de la devolución y se informa como tal.
